# Testing Notebook — ApplianceRecommender (V2)

This notebook is meant to be run from inside your project (so the relative imports in
`recommender.py` resolve correctly). It tests the three V2 improvements one at a time:

1. Capacity tolerance filtering
2. Deal score (replacing value score)
3. Brand diversity control

**Before running**: check the import in the next cell. Right now it assumes `recommender.py`
lives somewhere importable as a module — adjust the path/import to match where you actually
saved the updated file in your project.

In [ ]:
# --- Setup ---
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# NOTE: adjust this import to match where recommender.py actually lives in your project.
# Example: if recommender.py is at PROJECT_ROOT/app/recommender.py, this works as-is
# as long as you run the notebook from somewhere inside the project.
from recommender import ApplianceRecommender

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

## Step 1 — Initialize the Recommender

This loads your trained model, the X_train/y_train parquet files, and pre-computes
`fair_price` and `deal_score` for the whole pool. If this cell fails, the problem is almost
always a path issue (model file or parquet file not found) — fix that before testing
anything else, since every test below depends on this object loading correctly.

In [ ]:
recommender = ApplianceRecommender()

print("Pool shape:", recommender.pool.shape)
print("Pool columns:", list(recommender.pool.columns))
print()
print("Tolerance map:", recommender.tolerance_map)
print("Max per brand:", recommender.max_per_brand)

## Step 2 — Helper Function to Display Results

`get_recommendations()` returns a list of dicts. This helper turns that into a clean
DataFrame and shows `deal_score` as a percentage instead of a raw decimal, since that's
the whole point of Improvement 2 — it should be easy to read at a glance.

In [ ]:
def show_results(results, capacity_col=None):
    """Convert get_recommendations() output into a readable DataFrame."""
    if not results:
        print("No results returned (empty list).")
        return pd.DataFrame()

    df = pd.DataFrame(results)

    # Show deal_score as a percentage string for readability
    df['deal_score_pct'] = (df['deal_score'] * 100).round(1).astype(str) + '%'

    cols_to_show = ['brand_name', 'category']
    if capacity_col and capacity_col in df.columns:
        cols_to_show.append(capacity_col)
    cols_to_show += ['actual_price', 'fair_price', 'deal_score_pct', 'recommendation_score']

    # Only keep columns that actually exist, in case schema differs slightly
    cols_to_show = [c for c in cols_to_show if c in df.columns]

    display(df[cols_to_show])
    return df

## Step 3 — Test: AC Recommendation

**Important**: the `user_input` dictionary below is a guess at the fields your
`build_feature_dataframe()` function expects, based on the column names visible in
`recommender.py` (`category`, `capacity_value`, brand-like fields, star rating, inverter).

Open `src/bi_layer.py` and check what `build_feature_dataframe()` actually reads from
`user_input`, then adjust the dictionary keys below to match exactly. If a required key is
missing, this will likely throw a `KeyError` or silently produce a badly encoded row.

In [ ]:
ac_input = {
    'category': 'AC',
    'capacity_value': 1.5,       # tons
    'brand_name': 'lg',
    'star_rating': 5,
    'inverter': True
}

ac_results = recommender.get_recommendations(ac_input, n=10)
ac_df = show_results(ac_results, capacity_col='capacity_ac_tons')

## Step 4 — Validate Capacity Tolerance (Improvement 1)

Every returned AC should have a capacity within `tolerance_map['AC']` (0.2 tons) of the
requested 1.5 tons. This cell checks that automatically instead of you eyeballing the table.

In [ ]:
if not ac_df.empty:
    tolerance = recommender.tolerance_map['AC']
    requested_capacity = ac_input['capacity_value']

    capacity_diffs = (ac_df['capacity_ac_tons'] - requested_capacity).abs()
    within_tolerance = (capacity_diffs <= tolerance).all()

    print(f"Requested capacity: {requested_capacity} tons | Tolerance: +/- {tolerance} tons")
    print("Max capacity difference in results:", capacity_diffs.max())
    print("PASS: all results within tolerance" if within_tolerance else "FAIL: some results outside tolerance")
else:
    print("No results to validate — check ac_input fields or widen the pool.")

## Step 5 — Validate Brand Diversity (Improvement 3)

No single brand should appear more than `max_per_brand` (2) times in the results. This
counts brand occurrences directly rather than just visually scanning the table above.

In [ ]:
if not ac_df.empty:
    brand_counts = ac_df['brand_name'].value_counts()
    print(brand_counts)
    print()

    max_allowed = recommender.max_per_brand
    diversity_ok = (brand_counts <= max_allowed).all()
    print(f"PASS: no brand exceeds {max_allowed}" if diversity_ok else f"FAIL: a brand exceeds {max_allowed}")
else:
    print("No results to validate.")

## Step 6 — Test: Refrigerator Recommendation

Same pattern as above, but for the Refrigerator category. Adjust `capacity_value` and
brand to whatever exists in your dataset (check `recommender.pool['brand_name'].unique()` if
you're not sure what brand strings look like).

In [ ]:
fridge_input = {
    'category': 'Refrigerator',
    'capacity_value': 240,       # litres
    'brand_name': 'samsung',
    'star_rating': 4,
}

fridge_results = recommender.get_recommendations(fridge_input, n=10)
fridge_df = show_results(fridge_results, capacity_col='capacity_ref_litres')

if not fridge_df.empty:
    tolerance = recommender.tolerance_map['Refrigerator']
    diffs = (fridge_df['capacity_ref_litres'] - fridge_input['capacity_value']).abs()
    print(f"\nMax capacity difference: {diffs.max()} (tolerance: {tolerance})")
    print(fridge_df['brand_name'].value_counts())

## Step 7 — Test: Washing Machine Recommendation

This one is worth watching closely. In the previous discussion, the 1 kg tolerance for
washing machines was flagged as potentially too loose if your dataset's capacity steps
are close together (e.g. 6.5, 7.0, 7.5 kg). Check the spread of `capacity_wm_kg` values
in the results below — if they span more than you'd expect a real customer to accept,
tighten `recommender.tolerance_map['Washing Machine']` to something like 0.5.

In [ ]:
wm_input = {
    'category': 'Washing Machine',
    'capacity_value': 7.0,       # kg
    'brand_name': 'ifb',
    'star_rating': 4,
}

wm_results = recommender.get_recommendations(wm_input, n=10)
wm_df = show_results(wm_results, capacity_col='capacity_wm_kg')

if not wm_df.empty:
    print("\nCapacity spread in results:")
    print(wm_df['capacity_wm_kg'].sort_values().to_string(index=False))

## Step 8 — Edge Cases

A good test always checks the boundaries, not just the happy path. These two cells check
that invalid input is handled gracefully (returns an empty list, not a crash).

In [ ]:
# Edge case 1: invalid category should return an empty list, not crash
invalid_input = {
    'category': 'Microwave',   # not in category_capacity_map
    'capacity_value': 20,
}
result = recommender.get_recommendations(invalid_input, n=5)
print("Invalid category result:", result)
print("PASS: empty list returned" if result == [] else "FAIL: expected empty list")

In [ ]:
# Edge case 2: a capacity value that doesn't exist anywhere near the dataset
# should also return an empty list (no eligible products even with tolerance applied)
extreme_input = {
    'category': 'AC',
    'capacity_value': 999,   # no AC is 999 tons
}
result = recommender.get_recommendations(extreme_input, n=5)
print("Extreme capacity result:", result)
print("PASS: empty list returned" if result == [] else "FAIL: expected empty list")

## Step 9 — Sanity Check on Deal Score Distribution

`deal_score = (fair_price - actual_price) / fair_price`. A healthy distribution should
mostly sit between roughly -0.3 and +0.3 (i.e. no product is wildly more than 30%
overpriced or underpriced relative to the model's prediction). If you see scores far
outside that range, it's worth checking whether the underlying XGBoost price model has
outlier predictions for those specific rows.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(recommender.pool['deal_score'], bins=40)
plt.axvline(0, color='red', linestyle='--', label='Fair price (0%)')
plt.xlabel('Deal Score')
plt.ylabel('Number of Products')
plt.title('Distribution of Deal Score Across Entire Pool')
plt.legend()
plt.show()

print(recommender.pool['deal_score'].describe())